# Run1 Standard (FIXED): MGSM Bengali-only — Qwen3.5-9B Q4_K_M GGUF via lm-eval

**Model:** `psxog/SAHAYAKAI` / `Qwen3.5-9B.Q4_K_M.gguf` (~6GB Q4_K_M)

**Task:** `mgsm_native_cot_bn` (250 Bengali, 8-shot CoT native, MGSM paper standard, harness v4.0, dataset `juletxara/mgsm`)

**Backend:** `gguf` via `llama-server` (`base_url=http://127.0.0.1:8081`, completion mode, NO chat template — leaderboard-comparable)

**Decoding:** greedy (`do_sample=false, temp=0.0`), `max_gen_toks=256`

**Output:** `/content/mgsm_bn_results/results.json` + `samples_mgsm_native_cot_bn.jsonl`

Fixes applied: correct task ID (`mgsm_native_cot_bn`, not `mgsm_cot_native_bn`), new CLI (`lm_eval run`, `--show_config`, `validate`), server-based `gguf` backend (`base_url`, not `pretrained=`), port `8081` (Colab holds `8080`), `parallel 1` for T4.

## 1. Setup — Install & GPU check

In [ ]:
!nvidia-smi
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q "lm-eval>=0.4.13" huggingface_hub
# Fallback if cu121 404s:
# !pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
!curl -LsSf https://llama.app/install.sh | sh
!export PATH="$HOME/.local/bin:$PATH"; ~/.local/bin/llama --version
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"
!lm-eval ls tasks 2>&1 | grep -i "mgsm.*bn"
import sys
print("Python", sys.version)

## 2. Download model

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = "psxog/SAHAYAKAI"
filename = "Qwen3.5-9B.Q4_K_M.gguf"
print(f"Downloading {repo_id}/{filename} ...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Model: {model_path} ({os.path.getsize(model_path)/1e9:.2f} GB)")
os.environ["MODEL_PATH"] = model_path
%env MODEL_PATH={model_path}

## 3. Start llama-server on 8081 (current `gguf` backend is server-based, needs `base_url`)

In [ ]:
import os
print("MODEL_PATH:", os.environ.get("MODEL_PATH"))
# NOTE: binary is ~/.local/bin/llama (new wrapper), NOT bare llama-server; 8080 is often held in Colab, so use 8081.
!pkill -9 -f llama 2>/dev/null; sleep 2; echo "killed old server"
!export PATH="$HOME/.local/bin:$PATH"; nohup ~/.local/bin/llama serve -m "$MODEL_PATH" -c 4096 --port 8081 -ngl -1 --parallel 1 > /tmp/llama8081.log 2>&1 &
!sleep 10; curl -s http://127.0.0.1:8081/health; echo; tail -n 20 /tmp/llama8081.log
# Expect first: 503 Loading model (normal). If 'couldn't bind': port busy -> check ss/lsof, or switch port.

## 4. Wait until server ready (9B Q4 load takes 30-90s; 503 + blk.32 warnings are normal)

In [ ]:
!for i in $(seq 1 12); do echo -n "$i: "; curl -s http://127.0.0.1:8081/health; echo; sleep 10; done
!tail -n 10 /tmp/llama8081.log
# Proceed only when a poll returns {"status":"ok"} + log shows 'model loaded' / 'listening on http://127.0.0.1:8081'.

## 5. Verify task (expect mgsm_native_cot_bn, mgsm_direct_bn, mgsm_en_cot_bn)

In [ ]:
!lm-eval ls tasks 2>&1 | grep -i "mgsm.*bn"
!lm-eval validate --tasks mgsm_native_cot_bn
# Expect: dataset juletxara/mgsm bn, doc_to_text contains ধাপে ধাপে উত্তর:,
# until ['প্রশ্ন:','</s>','<|im_end|>'], do_sample=false, temp=0.0, version 4.0

## 6. Smoke-test completions endpoint (must return choices before eval)

In [ ]:
!curl -s -X POST http://127.0.0.1:8081/v1/completions -H "Content-Type: application/json" -d '{"prompt":"1+1=","max_tokens":5}' | head -c 300; echo

## 7. Run eval — Bengali only, Run1 standard (no chat template, greedy, max_gen_toks=256)

In [ ]:
# Server-based gguf backend (harness>=0.4.9): pass base_url, NOT pretrained=...
# No --apply_chat_template / --system_instruction -> completion mode (standard).
!lm_eval run --model gguf \
  --model_args base_url=http://127.0.0.1:8081,max_gen_toks=256,timeout=300 \
  --tasks mgsm_native_cot_bn \
  --num_fewshot 8 \
  --batch_size 1 \
  --output_path /content/mgsm_bn_results \
  --log_samples \
  --show_config

## 8. Inspect results

In [ ]:
import json, os, glob
res_path = "/content/mgsm_bn_results/results.json"
print(open(res_path).read()[:4000])
print("\n--- files ---")
print(glob.glob("/content/mgsm_bn_results/*"))
# Metrics: exact_match strict-match + flexible-extract. Truncation at exactly 256 -> Run2 with 512 (label non-standard).


## 9. Download

In [ ]:
from google.colab import files
import os
for p in ["/content/mgsm_bn_results/results.json"]:
    if os.path.exists(p):
        print(f"Ready: {p} ({os.path.getsize(p)/1e3:.1f} KB)")
# files.download("/content/mgsm_bn_results/results.json")
# !ls /content/mgsm_bn_results/samples* 2>/dev/null && echo 'download samples file too'